# Streaming

> **提示**
>
> 对于新应用, 我们推荐 [事件 streaming](https://docs.langchain.com/oss/langgraph/event-streaming) — 即 LangGraph v1.2 引入的类型化投影 API。事件 streaming 为每个投影(messages、values、subgraphs、output)提供独立的迭代器, 因此你可以独立消费它们, 而不必根据 `stream_mode` 分块进行分支判断。

本页介绍 LangGraph 的 stream-mode API。它通过 `updates`、`values`、`messages`、`custom`、`checkpoints`、`tasks` 与 `debug` 等 stream mode 暴露 graph 执行。当你需要直接访问 graph-runtime 事件或特定 stream mode 输出时, 可以使用它。

In [1]:
# 环境准备: 从 .env 加载 API keys
from dotenv import load_dotenv

load_dotenv()  # 自动向上查找 src/oss/langgraph/.env

# ---- 统一导入 ----
import json
import operator
import os
from dataclasses import dataclass
from typing import Annotated, Any, TypedDict

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import MemorySaver
from langgraph.config import get_stream_writer
from langgraph.graph import END, START, StateGraph
from langgraph.types import StreamWriter
from openai import AsyncOpenAI

## 快速开始

### 基本用法

LangGraph 的 graph 暴露 `stream` (同步)与 `astream` (async)方法, 以迭代器的形式产出 stream 输出。传入一个或多个 [stream mode](#stream-mode) 来控制你收到的数据。

In [2]:
class State(TypedDict):
    topic: str
    joke: str


def generate_joke(state: State):
    writer = get_stream_writer()
    writer({"status": "thinking of a joke..."})
    return {"joke": f"Why did the {state['topic']} go to school? To get a sundae education!"}


graph = (
    StateGraph(State)
    .add_node(generate_joke)
    .add_edge(START, "generate_joke")
    .add_edge("generate_joke", END)
    .compile()
)

for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode=["updates", "custom"],  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            print(f"Node {node_name} updated: {state}")
    elif chunk["type"] == "custom":
        print(f"Status: {chunk['data']['status']}")

Status: thinking of a joke...
Node generate_joke updated: {'joke': 'Why did the ice cream go to school? To get a sundae education!'}


```shell title="Output"
Status: thinking of a joke...
Node generate_joke updated: {'joke': 'Why did the ice cream go to school? To get a sundae education!'}
```

### Stream 输出格式 (v2)

> **注意**
>
> 需要 LangGraph 1.1 或更高版本。本页所有示例都使用 `version="v2"`。

向 `stream()` 或 `astream()` 传入 `version="v2"` 可获得统一的输出格式。每个 chunk 都是一个 `StreamPart` dict, 形状一致 — 无论 stream mode、mode 数量或 subgraph 设置如何:

```python
{
    "type": "values" | "updates" | "messages" | "custom" | "checkpoints" | "tasks" | "debug",
    "ns": (),           # namespace tuple, populated for subgraph events
    "data": ...,        # the actual payload (type varies by stream mode)
}
```

每个 stream mode 都有一个对应的 `TypedDict`, 分别包含 `ValuesStreamPart`、`UpdatesStreamPart`、`MessagesStreamPart`、`CustomStreamPart`、`CheckpointStreamPart`、`TasksStreamPart`、`DebugStreamPart`。你可以从 `langgraph.types` 导入这些类型。联合类型 `StreamPart` 是基于 `part["type"]` 的不相交联合 (disjoint union), 让你在编辑器和类型检查器中获得完整的类型收窄。

在 v1(默认)中, 输出格式会根据你的 streaming 选项而变化(单一 mode 返回原始数据, 多个 mode 返回 `(mode, data)` 元组, subgraph 返回 `(namespace, data)` 元组)。在 v2 中, 格式始终一致:

**v2 (new)**

```python
for chunk in graph.stream(inputs, stream_mode="updates", version="v2"):
    print(chunk["type"])  # "updates"
    print(chunk["ns"])    # ()
    print(chunk["data"])  # {"node_name": {"key": "value"}}
```

**v1 (current default)**

```python
for chunk in graph.stream(inputs, stream_mode="updates"):
    print(chunk)  # {"node_name": {"key": "value"}}
```

v2 格式还支持类型收窄, 这意味着你可以按 `chunk["type"]` 过滤 chunk, 并获得正确的 payload 类型。每个分支都会把 `part["data"]` 收窄为该 mode 对应的具体类型:

```python
for part in graph.stream(
    {"topic": "ice cream"},
    stream_mode=["values", "updates", "messages", "custom"],
    version="v2",
):
    if part["type"] == "values":
        # ValuesStreamPart — full state snapshot after each step
        print(f"State: topic={part['data']['topic']}")
    elif part["type"] == "updates":
        # UpdatesStreamPart — only the changed keys from each node
        for node_name, state in part["data"].items():
            print(f"Node `{node_name}` updated: {state}")
    elif part["type"] == "messages":
        # MessagesStreamPart — (message_chunk, metadata) from LLM calls
        msg, metadata = part["data"]
        print(msg.content, end="", flush=True)
    elif part["type"] == "custom":
        # CustomStreamPart — arbitrary data from get_stream_writer()
        print(f"Progress: {part['data']['progress']}%")
```

## Stream mode

把以下 stream mode 中的一个或多个以列表形式传给 `stream` 或 `astream` 方法:

| Mode | Type | 说明 |
| :--- | :--- | :---------- |
| [values](#graph-state) | `ValuesStreamPart` | 每一步之后的完整 state。 |
| [updates](#graph-state) | `UpdatesStreamPart` | 每一步之后的 state 更新。同一步中的多个更新会分别 stream。 |
| [messages](#llm-token) | `MessagesStreamPart` | 来自 LLM 调用的 (LLM token, metadata) 二元组。 |
| [custom](#custom-data) | `CustomStreamPart` | node 通过 `get_stream_writer` 发出的自定义数据。 |
| [checkpoints](#checkpoint) | `CheckpointStreamPart` | checkpoint 事件(格式与 `get_state()` 相同)。需要 checkpointer。 |
| [tasks](#task) | `TasksStreamPart` | 带结果与错误的 task 启动/完成事件。需要 checkpointer。 |
| [debug](#debug) | `DebugStreamPart` | 所有可用信息 — 把 `checkpoints` 与 `tasks` 与额外 metadata 组合在一起。 |

### Graph state

使用 `updates` 与 `values` stream mode 在 graph 执行时 stream 其 state。

* `updates` 在 graph 每一步之后 stream state 的 **更新**。
* `values` 在 graph 每一步之后 stream state 的 **完整值**。

In [3]:
class State(TypedDict):
    topic: str
    joke: str


def refine_topic(state: State):
    return {"topic": state["topic"] + " and cats"}


def generate_joke(state: State):
    return {"joke": f"This is a joke about {state['topic']}"}


graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .add_edge("generate_joke", END)
    .compile()
)

#### updates

用它只 stream 各 node 在每一步之后返回的 **state 更新**。stream 出的输出既包含 node 名称, 也包含该更新。

In [4]:
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="updates",  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            print(f"Node `{node_name}` updated: {state}")

Node `refine_topic` updated: {'topic': 'ice cream and cats'}
Node `generate_joke` updated: {'joke': 'This is a joke about ice cream and cats'}


```shell title="Output"
Node `refine_topic` updated: {'topic': 'ice cream and cats'}
Node `generate_joke` updated: {'joke': 'This is a joke about ice cream and cats'}
```

#### values

用它 stream graph 在每一步之后的 **完整 state**。

```python
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="values",  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "values":
        print(f"topic: {chunk['data']['topic']}, joke: {chunk['data']['joke']}")
```

```shell title="Output"
topic: ice cream, joke:
topic: ice cream and cats, joke:
topic: ice cream and cats, joke: This is a joke about ice cream and cats
```

### Debug

使用 `debug` streaming mode 在 graph 执行全程 stream 尽可能多的信息。stream 出的输出既包含 node 名称, 也包含完整 state。

In [5]:
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="debug",  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "debug":
        print(chunk["data"])

{'step': 1, 'timestamp': '2026-10-01T16:24:58.996950+00:00', 'type': 'task', 'payload': {'id': '77af435e-5ded-8d70-a904-b1d61db9290f', 'name': 'refine_topic', 'input': {'topic': 'ice cream'}, 'triggers': ('branch:to:refine_topic',)}}
{'step': 1, 'timestamp': '2026-10-01T16:24:58.997581+00:00', 'type': 'task_result', 'payload': {'id': '77af435e-5ded-8d70-a904-b1d61db9290f', 'name': 'refine_topic', 'error': None, 'result': {'topic': 'ice cream and cats'}, 'interrupts': []}}
{'step': 2, 'timestamp': '2026-10-01T16:24:58.997840+00:00', 'type': 'task', 'payload': {'id': '40a4ab3e-30f7-a647-5f0c-6ad4e4fad10a', 'name': 'generate_joke', 'input': {'topic': 'ice cream and cats'}, 'triggers': ('branch:to:generate_joke',)}}
{'step': 2, 'timestamp': '2026-10-01T16:24:58.998086+00:00', 'type': 'task_result', 'payload': {'id': '40a4ab3e-30f7-a647-5f0c-6ad4e4fad10a', 'name': 'generate_joke', 'error': None, 'result': {'joke': 'This is a joke about ice cream and cats'}, 'interrupts': []}}


> **注意**
>
> `debug` mode 把 `checkpoints` 与 `tasks` 事件与额外 metadata 组合在一起。如果你只需要 debug 信息的一个子集, 请直接使用 `checkpoints` 或 `tasks`。

### LLM token

使用 `messages` streaming mode, 可以从 graph 的任意部分 — node、tool、subgraph 或 task — 逐 **token** stream Large Language Model (LLM) 的输出。

`messages` mode 的 stream 输出是一个元组 `(message_chunk, metadata)`, 其中:

* `message_chunk`: 来自 LLM 的 token 或消息片段。
* `metadata`: 一个包含 graph node 与 LLM 调用相关详细信息的 dict。

> 如果你的 LLM 无法以 LangChain integration 的形式提供, 你可以改用 `custom` mode 来 stream 它的输出。详情见 [与任意 LLM 一起使用](#与任意-llm-一起使用)。

> **警告**
>
> **Python < 3.11 中的 async 需要手动传入 config**
>
> 在 Python < 3.11 中使用 async 代码时, 你必须向 `ainvoke()` 显式传入 `RunnableConfig`, 才能启用正确的 streaming。详情见 [Python < 3.11 中的 async](#python--311-中的-async), 或升级到 Python 3.11+。

In [6]:
@dataclass
class MyState:
    topic: str
    joke: str = ""


model = init_chat_model("deepseek:deepseek-chat")


def call_model(state: MyState):
    """Call the LLM to generate a joke about a topic"""
    # Note that message events are emitted even when the LLM is run using .invoke rather than .stream
    model_response = model.invoke(  # [!code highlight]
        [
            {"role": "user", "content": f"Generate a joke about {state.topic}"}
        ]
    )
    return {"joke": model_response.content}


graph = (
    StateGraph(MyState)
    .add_node(call_model)
    .add_edge(START, "call_model")
    .compile()
)

# The "messages" stream mode streams LLM tokens with metadata
# Use version="v2" for a unified StreamPart format
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="messages",  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "messages":
        message_chunk, metadata = chunk["data"]
        if message_chunk.content:
            print(message_chunk.content, end="|", flush=True)

Why|

 did|

 the|

 ice|

 cream|

...

 case|

 of|

 brain|

 freeze|

.|

#### 按 LLM 调用过滤

你可以把 `tags` 关联到 LLM 调用上, 从而按 LLM 调用过滤 stream 出的 token。

```python
from langchain.chat_models import init_chat_model

# model_1 is tagged with "joke"
model_1 = init_chat_model("deepseek:deepseek-chat", tags=['joke'])
# model_2 is tagged with "poem"
model_2 = init_chat_model("deepseek:deepseek-chat", tags=['poem'])

graph = ... # define a graph that uses these LLMs

# The stream_mode is set to "messages" to stream LLM tokens
# The metadata contains information about the LLM invocation, including the tags
async for chunk in graph.astream(
    {"topic": "cats"},
    stream_mode="messages",  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        # Filter the streamed tokens by the tags field in the metadata to only include
        # the tokens from the LLM invocation with the "joke" tag
        if metadata["tags"] == ["joke"]:
            print(msg.content, end="|", flush=True)
```

##### 扩展示例: 按 tags 过滤

In [7]:
class JokePoemState(TypedDict):
    topic: str
    joke: str
    poem: str


# The joke_model is tagged with "joke"
joke_model = init_chat_model("deepseek:deepseek-chat", tags=["joke"])
# The poem_model is tagged with "poem"
poem_model = init_chat_model("deepseek:deepseek-chat", tags=["poem"])


async def call_model(state, config):
    topic = state["topic"]
    print("Writing joke...")
    # Note: Passing the config through explicitly is required for python < 3.11
    # Since context var support wasn't added before then: https://docs.python.org/3/library/asyncio-task.html#creating-tasks
    # The config is passed through explicitly to ensure the context vars are propagated correctly
    # This is required for Python < 3.11 when using async code. Please see the async section for more details
    joke_response = await joke_model.ainvoke(
        [{"role": "user", "content": f"Write a joke about {topic}"}],
        config,
    )
    print("\n\nWriting poem...")
    poem_response = await poem_model.ainvoke(
        [{"role": "user", "content": f"Write a short poem about {topic}"}],
        config,
    )
    return {"joke": joke_response.content, "poem": poem_response.content}


graph = (
    StateGraph(JokePoemState)
    .add_node(call_model)
    .add_edge(START, "call_model")
    .compile()
)

# The stream_mode is set to "messages" to stream LLM tokens
# The metadata contains information about the LLM invocation, including the tags
async for chunk in graph.astream(
    {"topic": "cats"},
    stream_mode="messages",
    version="v2",
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        if metadata["tags"] == ["joke"]:
            print(msg.content, end="|", flush=True)

Writing joke...


|

Why|

 did|

 the|

...

ist|

!|

|

|



Writing poem...


#### 从 stream 中省略消息

使用 `nostream` tag 将 LLM 输出完全排除在 stream 之外。带有 `nostream` tag 的调用仍会运行并产生输出; 它们的 token 只是不会在 `messages` mode 中发出。

它在以下情况很有用:

- 你需要 LLM 输出用于内部处理(例如结构化输出), 但不希望把它 stream 给客户端
- 你通过另一个 channel 来 stream 相同内容(例如自定义 UI 消息), 并希望避免在 `messages` stream 中出现重复输出

In [8]:
class NotesState(TypedDict):
    topic: str
    answer: str
    notes: str


stream_model = init_chat_model("deepseek:deepseek-chat")
internal_model = init_chat_model("deepseek:deepseek-chat").with_config(
    {"tags": ["nostream"]}
)


def answer(state: NotesState) -> dict[str, Any]:
    r = stream_model.invoke(
        [{"role": "user", "content": f"Reply briefly about {state['topic']}"}]
    )
    return {"answer": r.content}


def internal_notes(state: NotesState) -> dict[str, Any]:
    # Tokens from this model are omitted from stream_mode="messages" because of nostream
    r = internal_model.invoke(
        [{"role": "user", "content": f"Private notes on {state['topic']}"}]
    )
    return {"notes": r.content}


graph = (
    StateGraph(NotesState)
    .add_node("write_answer", answer)
    .add_node("internal_notes", internal_notes)
    .add_edge(START, "write_answer")
    .add_edge("write_answer", "internal_notes")
    .compile()
)

initial_state: NotesState = {"topic": "AI", "answer": "", "notes": ""}
stream = graph.stream_events(initial_state, version="v3")

C:\Users\Ashit\Documents\GitHub\LangChain-Docs\.venv\Lib\site-packages\langgraph\pregel\main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
C:\Users\Ashit\Documents\GitHub\LangChain-Docs\.venv\Lib\site-packages\langgraph\pregel\main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


#### 按 node 过滤

若只想 stream 来自特定 node 的 token, 请使用 `stream_mode="messages"` 并在 stream 出的 metadata 中按 `langgraph_node` 字段过滤输出:

```python
# The "messages" stream mode streams LLM tokens with metadata
# Use version="v2" for a unified StreamPart format
for chunk in graph.stream(
    inputs,
    stream_mode="messages",  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        # Filter the streamed tokens by the langgraph_node field in the metadata
        # to only include the tokens from the specified node
        if msg.content and metadata["langgraph_node"] == "some_node_name":
            ...
```

##### 扩展示例: 从特定 node stream LLM token

In [9]:
class JokePoemState(TypedDict):
    topic: str
    joke: str
    poem: str


def write_joke(state: JokePoemState):
    topic = state["topic"]
    joke_response = model.invoke(
        [{"role": "user", "content": f"Write a joke about {topic}"}]
    )
    return {"joke": joke_response.content}


def write_poem(state: JokePoemState):
    topic = state["topic"]
    poem_response = model.invoke(
        [{"role": "user", "content": f"Write a short poem about {topic}"}]
    )
    return {"poem": poem_response.content}


graph = (
    StateGraph(JokePoemState)
    .add_node(write_joke)
    .add_node(write_poem)
    # write both the joke and the poem concurrently
    .add_edge(START, "write_joke")
    .add_edge(START, "write_poem")
    .compile()
)

# The "messages" stream mode streams LLM tokens with metadata
# Use version="v2" for a unified StreamPart format
for chunk in graph.stream(
    {"topic": "cats"},
    stream_mode="messages",  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        # Filter the streamed tokens by the langgraph_node field in the metadata
        # to only include the tokens from the write_poem node
        if msg.content and metadata["langgraph_node"] == "write_poem":
            print(msg.content, end="|", flush=True)

Soft|

 paws|

 on|

 the|

 windows|

...

 all|

 a|

 merry|

 dance|

.|

### 自定义数据

若要从 LangGraph 的 node 或 tool 内部发送 **用户自定义数据**, 请按以下步骤操作:

1. 使用 `get_stream_writer` 访问 stream writer 并发出自定义数据。
2. 在调用 `.stream()` 或 `.astream()` 时设置 `stream_mode="custom"`, 以便在 stream 中获取自定义数据。你可以组合多个 mode(例如 `["updates", "custom"]`), 但至少要有一个是 `"custom"`。

> **警告**
>
> **Python < 3.11 的 async 中无法使用 `get_stream_writer`**
>
> 在 Python < 3.11 上运行的 async 代码中, `get_stream_writer` 不会生效。
> 请改为在 node 或 tool 中添加 `writer` 参数并手动传入。
> 用法示例见 [Python < 3.11 中的 async](#python--311-中的-async)。

#### node

In [10]:
class QueryState(TypedDict):
    query: str
    answer: str


def node(state: QueryState):
    # Get the stream writer to send custom data
    writer = get_stream_writer()
    # Emit a custom key-value pair (e.g., progress update)
    writer({"custom_key": "Generating custom data inside node"})
    return {"answer": "some data"}


graph = (
    StateGraph(QueryState)
    .add_node(node)
    .add_edge(START, "node")
    .compile()
)

inputs = {"query": "example"}

# Set stream_mode="custom" to receive the custom data in the stream
for chunk in graph.stream(inputs, stream_mode="custom", version="v2"):
    if chunk["type"] == "custom":
        print(f"Custom event: {chunk['data']['custom_key']}")

Custom event: Generating custom data inside node


#### tool

```python
from langchain.tools import tool
from langgraph.config import get_stream_writer

@tool
def query_database(query: str) -> str:
    """Query the database."""
    # Access the stream writer to send custom data
    writer = get_stream_writer()  # [!code highlight]
    # Emit a custom key-value pair (e.g., progress update)
    writer({"data": "Retrieved 0/100 records", "type": "progress"})  # [!code highlight]
    # perform query
    # Emit another custom key-value pair
    writer({"data": "Retrieved 100/100 records", "type": "progress"})
    return "some-answer"


graph = ... # define a graph that uses this tool

# Set stream_mode="custom" to receive the custom data in the stream
for chunk in graph.stream(inputs, stream_mode="custom", version="v2"):
    if chunk["type"] == "custom":
        print(f"{chunk['data']['type']}: {chunk['data']['data']}")
```

### Subgraph 输出

若要把 [subgraph](https://docs.langchain.com/oss/langgraph/use-subgraphs) 的输出纳入 stream 输出, 你可以在父 graph 的 `.stream()` 方法中设置 `subgraphs=True`。这会把父 graph 与任意 subgraph 的输出都 stream 出来。

输出会以元组 `(namespace, data)` 的形式 stream, 其中 `namespace` 是一个元组, 表示指向调用 subgraph 的 node 的路径, 例如 `("parent_node:<task_id>", "child_node:<task_id>")`。

#### v2 (LangGraph 1.1 或更高版本)

使用 `version="v2"` 时, subgraph 事件使用相同的 `StreamPart` 格式。`ns` 字段标识来源:

```python
for chunk in graph.stream(
    {"foo": "foo"},
    subgraphs=True,  # [!code highlight]
    stream_mode="updates",
    version="v2", # [!code highlight]
):
    print(chunk["type"])  # "updates"
    print(chunk["ns"])    # () for root, ("node_name:<task_id>",) for subgraph
    print(chunk["data"])  # {"node_name": {"key": "value"}}
```

#### v1 (默认)

```python
for chunk in graph.stream(
    {"foo": "foo"},
    # Set subgraphs=True to stream outputs from subgraphs
    subgraphs=True,  # [!code highlight]
    stream_mode="updates",
):
    print(chunk)
```

> **注意**
>
> 这适用于所有 `stream_mode`, 包括 `"messages"`。像 `create_agent` 这样的 agent builder 返回的是 **已编译 graph**, 因此把它添加为一个 node 会使其变成 subgraph。若不设置 `subgraphs=True`, 父 graph 上的 `stream_mode="messages"` 不会发出内部 agent LLM 调用产生的 token chunk。直接调用 `agent.stream(...)` 则会, 这也是为什么这个问题常常只在包装之后才出现。

```python
from langchain.agents import create_agent
from langgraph.graph import END, START, StateGraph

graph = (
    StateGraph(State)
    .add_node("agent", create_agent(model, tools, state_schema=State))
    .add_edge(START, "agent")
    .add_edge("agent", END)
    .compile()
)

for chunk in graph.stream(
    {"messages": [{"role": "user", "content": "..."}]},
    stream_mode="messages",
    subgraphs=True,  # [!code highlight]
    version="v2",
):
    print(chunk["type"])  # "messages"
    print(chunk["ns"])    # () for root, ("agent:<task_id>",) for subgraph
    print(chunk["data"])  # (token, metadata)
```

##### 扩展示例: 从 subgraph 进行 stream

In [11]:
# Define subgraph
class SubgraphState(TypedDict):
    foo: str  # note that this key is shared with the parent graph state
    bar: str


def subgraph_node_1(state: SubgraphState):
    return {"bar": "bar"}


def subgraph_node_2(state: SubgraphState):
    return {"foo": state["foo"] + state["bar"]}


subgraph_builder = StateGraph(SubgraphState)
subgraph_builder.add_node(subgraph_node_1)
subgraph_builder.add_node(subgraph_node_2)
subgraph_builder.add_edge(START, "subgraph_node_1")
subgraph_builder.add_edge("subgraph_node_1", "subgraph_node_2")
subgraph = subgraph_builder.compile()


# Define parent graph
class ParentState(TypedDict):
    foo: str


def node_1(state: ParentState):
    return {"foo": "hi! " + state["foo"]}


builder = StateGraph(ParentState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", subgraph)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
graph = builder.compile()

for chunk in graph.stream(
    {"foo": "foo"},
    stream_mode="updates",
    # Set subgraphs=True to stream outputs from subgraphs
    subgraphs=True,  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "updates":
        if chunk["ns"]:
            print(f"Subgraph {chunk['ns']}: {chunk['data']}")
        else:
            print(f"Root: {chunk['data']}")

Root: {'node_1': {'foo': 'hi! foo'}}


Subgraph ('node_2:628f1f63-245c-bf55-4b71-cfba76af52dc',): {'subgraph_node_1': {'bar': 'bar'}}
Subgraph ('node_2:628f1f63-245c-bf55-4b71-cfba76af52dc',): {'subgraph_node_2': {'foo': 'hi! foobar'}}
Root: {'node_2': {'foo': 'hi! foobar'}}


```shell title="Output"
Root: {'node_1': {'foo': 'hi! foo'}}
Subgraph ('node_2:dfddc4ba-c3c5-6887-5012-a243b5b377c2',): {'subgraph_node_1': {'bar': 'bar'}}
Subgraph ('node_2:dfddc4ba-c3c5-6887-5012-a243b5b377c2',): {'subgraph_node_2': {'foo': 'hi! foobar'}}
Root: {'node_2': {'foo': 'hi! foobar'}}
```

**注意**, 我们收到的不只是 node 更新, 还收到了 namespace, 它告诉我们正在从哪个 graph(或 subgraph)进行 stream。

### Checkpoint

使用 `checkpoints` streaming mode 在 graph 执行时接收 checkpoint 事件。每个 checkpoint 事件的格式与 `get_state()` 的输出相同。需要 [checkpointer](https://docs.langchain.com/oss/langgraph/persistence)。

In [12]:
graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .add_edge("generate_joke", END)
    .compile(checkpointer=MemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

for chunk in graph.stream(
    {"topic": "ice cream"},
    config=config,
    stream_mode="checkpoints",  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "checkpoints":
        print(chunk["data"])

{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f1bdb4a-791e-67f0-bfff-1d2b239f2e46'}}, 'parent_config': None, 'values': {}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': 'b5360a81-9175-d587-0076-5a8bde2c51a8', 'name': '__start__', 'interrupts': (), 'state': None}]}
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f1bdb4a-7922-61b7-8000-174774abf75a'}}, 'parent_config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f1bdb4a-791e-67f0-bfff-1d2b239f2e46'}}, 'values': {'topic': 'ice cream'}, 'metadata': {'source': 'loop', 'step': 0, 'parents': {}}, 'next': ['refine_topic'], 'tasks': [{'id': '01dec374-ae56-a0ac-d2eb-8f87385e2710', 'name': 'refine_topic', 'interrupts': (), 'state': None}]}
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f1bdb4a-7925-65b8-8001-78507e70b9aa'}}, 'parent_config': {'

### Task

使用 `tasks` streaming mode 在 graph 执行时接收 task 启动与完成事件。task 事件包含正在运行的是哪个 node、其结果以及任何错误的信息。需要 [checkpointer](https://docs.langchain.com/oss/langgraph/persistence)。

In [13]:
# 复用上文已定义(带 checkpointer)的 graph, 此处不再重复构建
config = {"configurable": {"thread_id": "1"}}

for chunk in graph.stream(
    {"topic": "ice cream"},
    config=config,
    stream_mode="tasks",  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "tasks":
        print(chunk["data"])

{'id': '7fef4ebf-9b72-3d4f-b3ed-991367e00ab0', 'name': 'refine_topic', 'input': {'topic': 'ice cream', 'joke': 'This is a joke about ice cream and cats'}, 'triggers': ('branch:to:refine_topic',)}
{'id': '7fef4ebf-9b72-3d4f-b3ed-991367e00ab0', 'name': 'refine_topic', 'error': None, 'result': {'topic': 'ice cream and cats'}, 'interrupts': []}
{'id': '0e958698-b973-b6fc-79c2-f475618057a2', 'name': 'generate_joke', 'input': {'topic': 'ice cream and cats', 'joke': 'This is a joke about ice cream and cats'}, 'triggers': ('branch:to:generate_joke',)}
{'id': '0e958698-b973-b6fc-79c2-f475618057a2', 'name': 'generate_joke', 'error': None, 'result': {'joke': 'This is a joke about ice cream and cats'}, 'interrupts': []}


### 同时使用多个 mode

你可以把一个列表作为 `stream_mode` 参数传入, 以同时 stream 多个 mode。

使用 `version="v2"` 时, 每个 chunk 都是一个 `StreamPart` dict。使用 `chunk["type"]` 来区分不同 mode:

```python v2
for chunk in graph.stream(inputs, stream_mode=["updates", "custom"], version="v2"):
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            print(f"Node `{node_name}` updated: {state}")
    elif chunk["type"] == "custom":
        print(f"Custom event: {chunk['data']}")
```

```python v1
for mode, chunk in graph.stream(inputs, stream_mode=["updates", "custom"]):
    print(chunk)
```

## 高级

### 与任意 LLM 一起使用

你可以使用 `stream_mode="custom"` 从 **任意 LLM API** stream 数据 — 即使该 API **没有** 实现 LangChain chat model 接口。

这让你能接入原始的 LLM client, 或提供自身 streaming 接口的外部服务, 使 LangGraph 对定制场景高度灵活。

```python
from langgraph.config import get_stream_writer

def call_arbitrary_model(state):
    """Example node that calls an arbitrary model and streams the output"""
    # Get the stream writer to send custom data
    writer = get_stream_writer()  # [!code highlight]
    # Assume you have a streaming client that yields chunks
    # Generate LLM tokens using your custom streaming client
    for chunk in your_custom_streaming_client(state["topic"]):
        # Use the writer to send custom data to the stream
        writer({"custom_llm_chunk": chunk})  # [!code highlight]
    return {"result": "completed"}

graph = (
    StateGraph(State)
    .add_node(call_arbitrary_model)
    # Add other nodes and edges as needed
    .compile()
)
# Set stream_mode="custom" to receive the custom data in the stream
for chunk in graph.stream(
    {"topic": "cats"},
    stream_mode="custom",  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "custom":
        # The chunk data will contain the custom data streamed from the llm
        print(chunk["data"])
```

##### 扩展示例: stream 任意 chat model

本示例使用 DeepSeek 的 OpenAI 兼容接口作为一个"任意 chat model"; 通过 OpenAI SDK 直接流式读取 token, 再用 `get_stream_writer()` 把 token 作为自定义数据发出。

In [14]:
# 使用 DeepSeek 的 OpenAI 兼容接口作为"任意 chat model"
openai_client = AsyncOpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)
model_name = "deepseek-chat"


async def stream_tokens(model_name: str, messages: list[dict]):
    response = await openai_client.chat.completions.create(
        messages=messages, model=model_name, stream=True
    )
    role = None
    async for chunk in response:
        delta = chunk.choices[0].delta

        if delta.role is not None:
            role = delta.role

        if delta.content:
            yield {"role": role, "content": delta.content}


# this is our tool
async def get_items(place: str) -> str:
    """Use this tool to list items one might find in a place you're asked about."""
    writer = get_stream_writer()
    response = ""
    async for msg_chunk in stream_tokens(
        model_name,
        [
            {
                "role": "user",
                "content": (
                    "Can you tell me what kind of items "
                    f"i might find in the following place: '{place}'. "
                    "List at least 3 such items separating them by a comma. "
                    "And include a brief description of each item."
                ),
            }
        ],
    ):
        response += msg_chunk["content"]
        writer(msg_chunk)

    return response


class MessagesState(TypedDict):
    messages: Annotated[list[dict], operator.add]


# this is the tool-calling graph node
async def call_tool(state: MessagesState):
    ai_message = state["messages"][-1]
    tool_call = ai_message["tool_calls"][-1]

    function_name = tool_call["function"]["name"]
    if function_name != "get_items":
        raise ValueError(f"Tool {function_name} not supported")

    function_arguments = tool_call["function"]["arguments"]
    arguments = json.loads(function_arguments)

    function_response = await get_items(**arguments)
    tool_message = {
        "tool_call_id": tool_call["id"],
        "role": "tool",
        "name": function_name,
        "content": function_response,
    }
    return {"messages": [tool_message]}


graph = (
    StateGraph(MessagesState)
    .add_node(call_tool)
    .add_edge(START, "call_tool")
    .compile()
)

让我们用一个包含 tool call 的 `AIMessage` 来调用 graph:

In [15]:
inputs = {
    "messages": [
        {
            "content": None,
            "role": "assistant",
            "tool_calls": [
                {
                    "id": "1",
                    "function": {
                        "arguments": '{"place":"bedroom"}',
                        "name": "get_items",
                    },
                    "type": "function",
                }
            ],
        }
    ]
}

async for chunk in graph.astream(
    inputs,
    stream_mode="custom",
    version="v2",
):
    if chunk["type"] == "custom":
        print(chunk["data"]["content"], end="|", flush=True)

Sure|

!|

 Here|

 are|

 three|

...

|

 you

|

 sleep|

.|

### 为特定 chat model 禁用 streaming

如果你的应用混用了支持 streaming 的 model 与不支持的 model, 你可能需要为不支持 streaming 的 model 显式禁用 streaming。

在初始化 model 时设置 `streaming=False`。

#### init_chat_model

In [16]:
model = init_chat_model(
    "deepseek:deepseek-chat",
    # Set streaming=False to disable streaming for the chat model
    streaming=False,  # [!code highlight]
)

#### Chat model 接口

In [17]:
# Set streaming=False to disable streaming for the chat model
model = ChatDeepSeek(model="deepseek-chat", streaming=False)

> **注意**
>
> 并非所有 chat model integration 都支持 `streaming` 参数。如果你的 model 不支持, 请改用 `disable_streaming=True`。所有 chat model 都可以通过基类使用该参数。

### 迁移到 v2

v2 streaming 格式(本页通篇使用)提供了统一的输出格式。以下是关键差异以及如何迁移的摘要:

| 场景 | v1(默认) | v2(`version="v2"`) |
|---|---|---|
| 单一 stream mode | 原始数据(dict) | 带 `type`、`ns`、`data` 的 `StreamPart` dict |
| 多个 stream mode | `(mode, data)` 元组 | 相同的 `StreamPart` dict, 按 `chunk["type"]` 过滤 |
| Subgraph streaming | `(namespace, data)` 元组 | 相同的 `StreamPart` dict, 检查 `chunk["ns"]` |
| 多个 mode + subgraph | `(namespace, mode, data)` 三元组 | 相同的 `StreamPart` dict |
| `invoke()` 返回类型 | 普通 dict(state) | 带 `.value` 与 `.interrupts` 的 `GraphOutput` |
| Interrupt 位置(stream) | state dict 中的 `__interrupt__` 键 | `values` stream part 上的 `interrupts` 字段 |
| Interrupt 位置(invoke) | result dict 中的 `__interrupt__` 键 | `GraphOutput` 上的 `.interrupts` 属性 |
| Pydantic/dataclass 输出 | 返回普通 dict | 强制转换为 model/dataclass 实例 |

#### v2 invoke 格式

当你向 `invoke()` 或 `ainvoke()` 传入 `version="v2"` 时, 它会返回一个带 `.value` 与 `.interrupts` 属性的 `GraphOutput` 对象:

```python
from langgraph.types import GraphOutput

result = graph.invoke(inputs, version="v2")

assert isinstance(result, GraphOutput)
result.value       # your output — dict, Pydantic model, or dataclass
result.interrupts  # tuple[Interrupt, ...], empty if none occurred
```

当使用除默认 `"values"` 之外的任意 stream mode 时, `invoke(..., stream_mode="updates", version="v2")` 返回 `list[StreamPart]`, 而不是 `list[tuple]`。

> **警告**
>
> 为向后兼容, 对 `GraphOutput` 的 dict 风格访问(`result["key"]`、`"key" in result`、`result["__interrupt__"]`)仍然可用, 但已被 **弃用**, 并会在未来版本中移除。请迁移到 `result.value` 与 `result.interrupts`。

这把 state 与 interrupt metadata 分离。在 v1 中, interrupt 被嵌入在返回 dict 的 `__interrupt__` 下:

**v2 (new)**

```python
config = {"configurable": {"thread_id": "thread-1"}}
result = graph.invoke(inputs, config=config, version="v2")

if result.interrupts:
    print(result.interrupts[0].value)
    graph.invoke(Command(resume=True), config=config, version="v2")
```

**v1 (current default)**

```python
config = {"configurable": {"thread_id": "thread-1"}}
result = graph.invoke(inputs, config=config)

if "__interrupt__" in result:
    print(result["__interrupt__"][0].value)
    graph.invoke(Command(resume=True), config=config)
```

#### Pydantic 与 dataclass 的 state 强制转换

当你的 graph state 是 Pydantic model 或 dataclass 时, v2 的 `values` mode 会自动把输出强制转换为正确的类型:

```python
from pydantic import BaseModel
from typing import Annotated
import operator

class MyState(BaseModel):
    value: str
    items: Annotated[list[str], operator.add]

# With version="v2", chunk["data"] is a MyState instance
for chunk in graph.stream(
    {"value": "x", "items": []}, stream_mode="values", version="v2"
):
    print(type(chunk["data"]))  # <class 'MyState'>
```

### Python < 3.11 中的 async

在 Python < 3.11 的版本中, [asyncio task](https://docs.python.org/3/library/asyncio-task.html#asyncio.create_task) 不支持 `context` 参数。
这限制了 LangGraph 自动传播 context 的能力, 并以两种关键方式影响 LangGraph 的 streaming 机制:

1. 你 **必须** 把 [`RunnableConfig`](https://python.langchain.com/docs/concepts/runnables/#runnableconfig) 显式传入 async LLM 调用(例如 `ainvoke()`), 因为 callback 不会被自动传播。
2. 你 **不能** 在 async node 或 tool 中使用 `get_stream_writer` — 你必须直接传入 `writer` 参数。

##### 扩展示例: 手动传入 config 的 async LLM 调用

In [18]:
class State(TypedDict):
    topic: str
    joke: str


model = init_chat_model("deepseek:deepseek-chat")


# Accept config as an argument in the async node function
async def call_model(state, config):
    topic = state["topic"]
    print("Generating joke...")
    # Pass config to model.ainvoke() to ensure proper context propagation
    joke_response = await model.ainvoke(  # [!code highlight]
        [{"role": "user", "content": f"Write a joke about {topic}"}],
        config,
    )
    return {"joke": joke_response.content}


graph = (
    StateGraph(State)
    .add_node(call_model)
    .add_edge(START, "call_model")
    .compile()
)

# Set stream_mode="messages" to stream LLM tokens
async for chunk in graph.astream(
    {"topic": "ice cream"},
    stream_mode="messages",  # [!code highlight]
    version="v2",  # [!code highlight]
):
    if chunk["type"] == "messages":
        message_chunk, metadata = chunk["data"]
        if message_chunk.content:
            print(message_chunk.content, end="|", flush=True)

Generating joke...


Why|

 did|

 the|

 ice|

...

 of|

 melt|

down|

s|

.|

##### 扩展示例: 使用 stream writer 的 async 自定义 streaming

In [19]:
class State(TypedDict):
    topic: str
    joke: str


# Add writer as an argument in the function signature of the async node or tool
# LangGraph will automatically pass the stream writer to the function
async def generate_joke(state: State, writer: StreamWriter):  # [!code highlight]
    writer({"custom_key": "Streaming custom data while generating a joke"})
    return {"joke": f"This is a joke about {state['topic']}"}


graph = (
    StateGraph(State)
    .add_node(generate_joke)
    .add_edge(START, "generate_joke")
    .compile()
)

# Set stream_mode="custom" to receive the custom data in the stream  # [!code highlight]
async for chunk in graph.astream(
    {"topic": "ice cream"},
    stream_mode="custom",
    version="v2",
):
    if chunk["type"] == "custom":
        print(chunk["data"])

{'custom_key': 'Streaming custom data while generating a joke'}
